# Regression: Redemption Queue Sensitivity to stETH Discount

Dependent variable: daily stETH redemption volume as a proportion of total outstanding stETH  
Key regressor: lowest hourly stETH/ETH discount on each day  
Controls: year fixed effects

In [ ]:
import stata_setup
stata_setup.config('/home/yichen', 'mp')

In [ ]:
%%stata

clear all
set more off

global PROCESSED_DATA "../../processed_data"
global TABLES         "../../lido-bank/tables"

In [ ]:
%%stata
* Install required packages (safe to re-run)
capture ssc install estout, replace

In [ ]:
%%stata
****************************
* Load and inspect data
****************************

import delimited using "$PROCESSED_DATA/reg_queue_discount.csv", varnames(1) clear

gen date2 = date(date, "YMD")
format date2 %td

* Scale redemption to basis points for readability (1 bp = 0.0001)
gen redemption_bp = redemption_pct * 10000

label var redemption_bp      "Redemption volume (bp of outstanding)"
label var min_steth_price    "Min daily stETH price (USD)"
label var min_discount_pct   "Min daily stETH/ETH discount (%)"
label var year               "Year"

summarize redemption_bp min_steth_price min_discount_pct year

In [ ]:
%%stata
****************************
* Regressions
* Y = daily stETH redemption requests / outstanding stETH (basis points)
* Primary X = min hourly stETH price on each day
* Alt X = min hourly stETH/ETH discount on each day
****************************

* (1) Min price only
eststo m1: quietly reg redemption_bp min_steth_price, robust

* (2) Min price + year FE  [primary spec]
eststo m2: quietly xi: reg redemption_bp min_steth_price i.year, robust

* (3) Discount + year FE  [alternative]
eststo m3: quietly xi: reg redemption_bp min_discount_pct i.year, robust

* (4) Both + year FE
eststo m4: quietly xi: reg redemption_bp min_steth_price min_discount_pct i.year, robust

esttab m1 m2 m3 m4, ///
    keep(min_steth_price min_discount_pct) ///
    b(%9.4f) se(%9.4f) ///
    star(* 0.10 ** 0.05 *** 0.01) ///
    stats(N r2 r2_a, labels("N" "R2" "Adj. R2") fmt(%9.0f %9.3f %9.3f)) ///
    mtitles("(1)" "(2)" "(3)" "(4)") ///
    indicate("Year FE = _Iyear_*") ///
    nonumbers nogaps

In [ ]:
%%stata
****************************
* Export LaTeX table (full output; Python cell below strips float wrapper)
****************************

esttab m1 m2 m3 m4 using "$TABLES/reg_queue_discount.tex", replace ///
    keep(min_steth_price min_discount_pct) ///
    varlabels(min_steth_price  "Min daily stETH price (USD)" ///
              min_discount_pct "Min daily discount (\%/ETH)") ///
    b(%9.4f) se(%9.4f) ///
    star(* 0.10 ** 0.05 *** 0.01) ///
    stats(N r2 r2_a, labels("Observations" "\$R^2\$" "Adj.\ \$R^2\$") fmt(%9.0f %9.3f %9.3f)) ///
    mtitles("(1)" "(2)" "(3)" "(4)") ///
    indicate("Year FE = _Iyear_*") ///
    booktabs nonumbers nonote collabels(none) nogaps

display "Full table written to $TABLES/reg_queue_discount.tex"

In [ ]:
import re, pathlib

path = pathlib.Path("../../lido-bank/tables/reg_queue_discount.tex")
raw  = path.read_text()

# Extract everything from \begin{tabular} through \end{tabular} (inclusive)
m = re.search(r"(\\begin\{tabular\}.*?\\end\{tabular\})", raw, re.DOTALL)
if not m:
    raise ValueError("Could not find \\begin{tabular}...\\end{tabular} in the file")

# esttab always emits this fixed definition — hardcode to avoid nested-brace regex issues
sym_def = r"\def\sym#1{\ifmmode^{#1}\else\(^{#1}\)\fi}" + "\n"

path.write_text(sym_def + m.group(1) + "\n")
print("Stripped to tabular-only. Preview:")
print(path.read_text()[:400])